# patchworld

> Patch graph world with crowding and moving resources (Phase 6).

Spec: `specs/multiagent/phase6/`. Layouts (`ring`, `torus`), `TeamPatchWorld` with fixed or logistic patches, reward schemes, and ring methods for circuit agents.

In [ ]:
#| default_exp patchworld


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from dataclasses import dataclass

import numpy as np

from explRL.circuit import ringActivity
from explRL.envs import ZoneArena
from explRL.multi import spawnRngs
from explRL.team_refs import yieldModel


@dataclass
class Layout:
    name: str
    pos: np.ndarray
    dist: np.ndarray
    T: np.ndarray


def ringLayout(N=32, travelScale=0.1) -> Layout:
    idx = np.arange(N)
    dist = np.minimum(np.abs(idx[:, None] - idx), N - np.abs(idx[:, None] - idx))
    return Layout('ring', idx, dist, dist * travelScale)


def torusLayout(side=8, travelScale=0.5) -> Layout:
    pos = np.array([[x, y] for y in range(side) for x in range(side)], float)
    L = side * side
    dist = np.zeros((L, L))
    for j in range(L):
        for k in range(L):
            dx = min(abs(pos[j, 0] - pos[k, 0]), side - abs(pos[j, 0] - pos[k, 0]))
            dy = min(abs(pos[j, 1] - pos[k, 1]), side - abs(pos[j, 1] - pos[k, 1]))
            dist[j, k] = np.hypot(dx, dy)
    return Layout('torus', pos, dist, dist * travelScale)


## World

In [ ]:
#| export
class TeamPatchWorld:
    def __init__(self, layout, nPatches=4, nDanger=4, model='fixed', intake=(0.5, 1.0, 1.5, 2.0),
                 K=100.0, g=0.2, Smin=1.0, c=1.0, tStay=1.0, dangerCost=1.0, Tmove=120.0, noise=0.0,
                 scheme='individual', symmetric=False, patchSites=None, dangerSites=None):
        self.layout, self.L = layout, layout.T.shape[0]
        self.nPatches, self.nDanger, self.model = int(nPatches), int(nDanger), model
        self.intake = np.broadcast_to(np.asarray(intake, float), (self.nPatches,)).copy()
        self.K, self.g, self.Smin, self.c = float(K), float(g), float(Smin), float(c)
        self.tStay, self.dangerCost, self.Tmove, self.noise = float(tStay), float(dangerCost), Tmove, float(noise)
        self.scheme, self.symmetric = scheme, bool(symmetric)
        self._patchSites, self._dangerSites = patchSites, dangerSites
        self.site = self.danger = self.stock = self.loc = self.stay = self.last = None
        self.tWorld = self.nextMove = self.nAgents = 0
        self.ledger = []
        self.rngSites = self.rngMoves = self.rngNoise = None

    def reset(self, seed, nAgents):
        self.rngSites, self.rngMoves, self.rngNoise = spawnRngs(seed, 3)
        L, P = self.L, self.nPatches
        self.site = np.array(self._patchSites if self._patchSites is not None else self.rngSites.choice(L, P, replace=False), int)
        if self._dangerSites is not None:
            self.danger = set(int(x) for x in self._dangerSites)
        else:
            free = [j for j in range(L) if j not in self.site]
            self.danger = set(int(x) for x in self.rngSites.choice(free, min(self.nDanger, len(free)), replace=False))
        self.stock = np.full(P, self.K) if self.model == 'logistic' else None
        self.loc = [int(x) for x in self.rngSites.integers(L, size=nAgents)]
        self.stay, self.last = [None] * nAgents, [(0.0, 0.0)] * nAgents
        self.tWorld, self.nextMove, self.nAgents, self.ledger = 0.0, (self.Tmove or None), nAgents, []

    def patchAt(self, loc):
        m = np.where(self.site == int(loc))[0]
        return int(m[0]) if len(m) else None

    def advance(self, t):
        if t < self.tWorld:
            raise ValueError('time went backwards')
        while self.Tmove and self.nextMove is not None and self.nextMove <= t:
            self._regrow(self.nextMove - self.tWorld)
            self.tWorld = self.nextMove
            self._move()
            self.nextMove += self.Tmove
        self._regrow(t - self.tWorld)
        self.tWorld = float(t)

    def _regrow(self, d):
        if self.model == 'logistic' and d > 0:
            S, K, g = self.stock, self.K, self.g
            self.stock = K / (1 + (K / S - 1) * np.exp(-g * d))

    def _move(self):
        for p in range(self.nPatches):
            free = [j for j in range(self.L) if j not in self.site and j not in self.danger]
            self.site[p] = int(self.rngMoves.choice(free))
        for i, st in enumerate(self.stay):
            if st and st[1] is not None:
                self.stay[i] = None

    def _observed(self, h):
        if self.noise <= 0 or h <= 0:
            return float(h)
        return float(h * max(0.0, 1 + self.noise * self.rngNoise.standard_normal()))

    def _claim(self, i, loc, s0, dur):
        p = self.patchAt(loc)
        self.stay[i] = (loc, p, s0, s0 + dur)
        if p is None:
            return 0.0, 0, None
        n = sum(1 for st in self.stay if st and st[1] == p and st[2] <= s0 < st[3])
        if self.model == 'fixed':
            ip = self.intake[p]
            h = (self.c if not np.isfinite(ip) else min(self.c, ip / n)) * dur
        else:
            h = min(self.c * dur, (self.stock[p] - self.Smin) / n)
            self.stock[p] -= h
        return float(h), n, p

    def _occ_patch(self, patch, t):
        return sum(1 for st in self.stay if st and st[1] == patch and st[2] <= t < st[3])

    def _reward(self, kind, p, n, obsH, danger, dur, t):
        if kind == 'travel':
            return 0.0
        if self.scheme == 'individual':
            return obsH - (self.dangerCost * dur if danger else 0.0)
        nDanger = sum(1 for st in self.stay if st and st[0] in self.danger and st[2] <= t < st[3])
        if self.scheme == 'team':
            tot = sum(yieldModel(self, q, self._occ_patch(q, t)) for q in range(self.nPatches))
            return (tot - self.dangerCost * nDanger) / self.nAgents * dur
        if danger:
            return -self.dangerCost * dur
        if p is None:
            return 0.0
        return (yieldModel(self, p, n) - yieldModel(self, p, n - 1)) * dur

    def apply(self, i, target, t):
        if not 0 <= target < self.L:
            raise ValueError(f'target {target} is not a location')
        self.advance(t)
        loc = self.loc[i]
        if target == loc:
            kind, dur = 'stay', self.tStay
            h, n, p = self._claim(i, loc, t, dur)
        else:
            kind, dur, self.stay[i], self.loc[i] = 'travel', float(self.layout.T[loc, target]), None, int(target)
            h, n, p = 0.0, 0, None
        danger = kind == 'stay' and loc in self.danger
        obsH = self._observed(h)
        reward = self._reward(kind, p, n, obsH, danger, dur, t)
        self.last[i] = (reward, obsH)
        s0, s1 = (t, t + dur) if kind == 'stay' else (float('nan'), float('nan'))
        self.ledger.append((i, t, dur, kind, self.loc[i], p, h, reward, s0, s1))
        return reward, dur, self.loc[i], dict(kind=kind, harvest=h, observed=obsH, nHere=n, patch=p, danger=danger, dur=dur)

    def observe(self, i):
        return dict(location=self.loc[i], travel=self.layout.T[self.loc[i]].copy(),
                    lastReward=self.last[i][0], lastHarvest=self.last[i][1])

    def neighbours(self, i, radius):
        loc = self.loc[i]
        return [k for k in range(self.nAgents) if k != i and self.layout.dist[loc, self.loc[k]] <= radius]

    def ringActivity(self, a):
        return ringActivity(a, self.L, self.symmetric)

    def valence(self, a):
        return 1 if self.patchAt(a) is not None else (-1 if a in self.danger else 0)

    def fixate(self, i, a, s0, dur, t):
        self.loc[i] = int(a)
        self.advance(t)
        h, n, p = self._claim(i, a, s0, dur)
        net = self._observed(h) - (self.dangerCost * dur if a in self.danger else 0.0)
        self.ledger.append((i, t, s0 - t + dur, 'iteration', a, p, h, float(np.sign(net)), s0, s0 + dur))
        return h, net

    @classmethod
    def zoneConfig(cls, symmetric=True, N=32, intake=np.inf, **kw):
        z = ZoneArena(N=N, symmetric=symmetric)
        safe = [a for a in range(N) if z.isSafe(a)]
        return cls(ringLayout(N), nPatches=len(safe), nDanger=N - len(safe), model='fixed', intake=intake,
                   Tmove=None, symmetric=symmetric, patchSites=safe,
                   dangerSites=[a for a in range(N) if not z.isSafe(a)], **kw)


## Tests

In [ ]:
from fastcore.test import test_eq, test_fail, test_close
from explRL.envs import ZoneArena

def tiny(**kw):
    "Ring of 8 locations, one patch at 0 (intake 1), no danger, no moves."
    args = dict(nPatches=1, nDanger=0, intake=(1.0,), patchSites=[0], dangerSites=[], Tmove=None, c=1.0)
    return TeamPatchWorld(ringLayout(8), **{**args, **kw})

def placed(w, locs, seed=0):
    "Reset w for len(locs) agents and put them at locs."
    w.reset(seed, len(locs))
    w.loc = list(locs)
    return w


In [ ]:
# AC-3 layouts: symmetric, zero diagonal, triangle inequality, ring steps, torus wrap
rng = np.random.default_rng(0)
for lay in (ringLayout(16), torusLayout(4)):
    d, T = lay.dist, lay.T
    test_eq(d, d.T); test_eq(np.diag(d), np.zeros(len(d))); test_close(T, T.T)
    for i, j, k in rng.integers(0, len(d), (50, 3)):
        assert d[i, k] <= d[i, j] + d[j, k] + 1e-9
ring = ringLayout(32)
test_eq(ring.dist[0, 5], 5); test_eq(ring.dist[10, 28], 14); test_close(ring.T[0, 5], 0.5)
tor = torusLayout(8)
cell = lambda x, y: y * 8 + x
test_eq(tor.dist[cell(0, 0), cell(7, 0)], 1.0)
test_eq(tor.dist[cell(0, 0), cell(0, 7)], 1.0)
test_close(tor.dist[cell(0, 0), cell(7, 7)], np.sqrt(2))

# AC-4 harvest: claimed at the start of a stay, split among the stays covering that time
w = placed(tiny(), [0, 0, 0])
test_close(w.apply(0, 0, 0.0)[3]['harvest'], 1.0)      # alone: min(c, I)
test_close(w.apply(1, 0, 0.0)[3]['harvest'], 0.5)      # second at the same start: min(c, I / 2)
test_close(w.apply(2, 0, 0.5)[3]['harvest'], 1 / 3)    # third while both stay: min(c, I / 3)
test_eq(w.apply(0, 0, 1.0)[3]['nHere'], 2)             # agent 1's stay ended at 1.0, agent 2's has not
w = placed(tiny(intake=np.inf, c=0.7), [0, 0, 0])
for i in range(3):
    test_close(w.apply(i, 0, 0.0)[3]['harvest'], 0.7)  # I = inf: constant claim c * tStay

w = placed(tiny(model='logistic'), [0])                 # regrowth: closed form vs fine Euler
w.stock[0] = 10.0
w.advance(5.0)
S, h = 10.0, 1e-5
for _ in range(round(5.0 / h)):
    S += w.g * S * (1 - S / w.K) * h
assert abs(w.stock[0] / S - 1) < 1e-6
w = placed(tiny(model='logistic'), [0] * 30)            # 30 claims drain the stock to its floor, never below
w.stock[0] = 3.0
for i in range(30):
    w.apply(i, 0, 0.0)
assert w.Smin <= w.stock[0] < w.Smin + 0.1

# travel lasts T[loc, j] and moves the agent; a stay lasts tStay; one ledger row per call
w = placed(tiny(), [1])
r, dt, loc, info = w.apply(0, 4, 0.0)
test_eq((r, dt, loc, info['kind']), (0.0, w.layout.T[1, 4], 4, 'travel'))
r, dt2, loc, info = w.apply(0, 4, dt)
test_eq((dt2, loc, info['kind']), (w.tStay, 4, 'stay'))
test_eq(len(w.ledger), 2)

# AC-5 moves exactly at multiples of Tmove, onto free non-danger locations, keeping stock; same seed, same moves
def mover(model='fixed', seed=3):
    return placed(TeamPatchWorld(ringLayout(16), nPatches=3, nDanger=3, model=model, intake=1.0, Tmove=10.0), [0], seed)
w = mover()
s0 = w.site.copy()
w.advance(9.999); test_eq(w.site, s0)
w.advance(10.0); assert (w.site != s0).all()
assert len(set(w.site.tolist())) == 3 and not set(w.site.tolist()) & w.danger
s10 = w.site.copy()
w.advance(19.9); test_eq(w.site, s10)
w.advance(35.0)                                          # moves at 20 and 30 in one call
w2 = mover(); w2.advance(35.0); test_eq(w2.site, w.site)
w = mover('logistic')
w.stock[:] = (20.0, 40.0, 60.0)
w.advance(10.0)
S = np.array([20.0, 40.0, 60.0])
test_close(w.stock, w.K / (1 + (w.K / S - 1) * np.exp(-w.g * 10.0)))

w = placed(tiny(), [0])
w.apply(0, 0, 1.0)
test_fail(lambda: w.apply(0, 0, 0.5), contains='backwards')
test_fail(lambda: w.apply(0, 8, 2.0), contains='not a location')
test_fail(lambda: w.apply(0, -1, 2.0), contains='not a location')
test_eq(len(w.ledger), 1)

# AC-6 hand-checked rewards. Patches at 0 (I = 3) and 4 (I = 1), danger at 6, c = 2, six agents, all at t = 0:
# two share patch 0, one alone at patch 4, one at danger, one at an empty location (2), one travels.
# E(0, 1) = 2, E(0, 2) = 3, E(1, 1) = 1; team = (sum of yields - danger count) / 6 after each stay registers.
expect = {'individual': [2, 1.5, 1, -1, 0, 0], 'difference': [2, 1, 1, -1, 0, 0],
          'team': [2 / 6, 3 / 6, 4 / 6, 3 / 6, 3 / 6, 0]}
for scheme, want in expect.items():
    w = placed(TeamPatchWorld(ringLayout(8), nPatches=2, intake=(3.0, 1.0), patchSites=[0, 4], dangerSites=[6],
                              c=2.0, Tmove=None, scheme=scheme), [0, 0, 4, 6, 2, 1])
    got = [w.apply(i, tgt, 0.0)[0] for i, tgt in enumerate([0, 0, 4, 6, 2, 3])]
    test_close(np.array(got), np.array(want, float))

# observations: noise 0 gives observed = true harvest; noise > 0 is reproducible by seed
w = placed(tiny(), [0])
info = w.apply(0, 0, 0.0)[3]
obs = w.observe(0)
test_eq(info['observed'], info['harvest']); test_eq(obs['lastHarvest'], info['harvest'])
test_eq(set(obs), {'location', 'travel', 'lastReward', 'lastHarvest'}); test_eq(obs['travel'], w.layout.T[0])
def noisy(seed):
    w = placed(tiny(noise=0.3), [0], seed)
    return [w.apply(0, 0, float(t))[3]['observed'] for t in range(5)]
test_eq(noisy(7), noisy(7)); assert noisy(7) != noisy(8) and noisy(7) != [1.0] * 5

# AC-3 neighbours: radius 0, 1 step, inf; an agent in transit counts at its destination
w = placed(tiny(), [0, 0, 1, 4])
test_eq(set(w.neighbours(0, 0)), {1}); test_eq(set(w.neighbours(0, 1)), {1, 2})
test_eq(set(w.neighbours(0, np.inf)), {1, 2, 3})
w.apply(3, 1, 0.0)
test_eq(set(w.neighbours(0, 1)), {1, 2, 3})

# zoneConfig mirrors ZoneArena: sites, valence and ring activity, both scenes
for sym in (True, False):
    z, w = ZoneArena(symmetric=sym), TeamPatchWorld.zoneConfig(symmetric=sym)
    w.reset(0, 1)
    test_eq(sorted(w.site.tolist()), [a for a in range(32) if z.isSafe(a)])
    test_eq(sorted(w.danger), [a for a in range(32) if not z.isSafe(a)])
    test_eq([w.valence(a) for a in range(32)], [z.valence(a) for a in range(32)])
    for a in range(32):
        test_eq(w.ringActivity(a), z.ringActivity(a))


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
